In [2]:
import pandas as pd
import numpy as np

# Load the datasets
print("Loading datasets...")
rpi_df = pd.read_csv('../data/HDBResalePriceIndex1Q2009100Quarterly.csv')
resale_df = pd.read_csv('../data/ResaleflatpricesbasedonregistrationdatefromJan2017onwards.csv')

print(f"RPI dataset shape: {rpi_df.shape}")
print(f"Resale prices dataset shape: {resale_df.shape}")

# Display first few rows of each dataset
print("\nHDB Resale Price Index (first 5 rows):")
print(rpi_df.head())
print("\nResale Flat Prices (first 5 rows):")
print(resale_df.head())

Loading datasets...
RPI dataset shape: (144, 2)
Resale prices dataset shape: (224541, 11)

HDB Resale Price Index (first 5 rows):
   quarter  index
0  1990-Q1   24.3
1  1990-Q2   24.4
2  1990-Q3   25.0
3  1990-Q4   24.7
4  1991-Q1   24.9

Resale Flat Prices (first 5 rows):
     month        town flat_type block        street_name storey_range  \
0  2017-01  ANG MO KIO    2 ROOM   406  ANG MO KIO AVE 10     10 TO 12   
1  2017-01  ANG MO KIO    3 ROOM   108   ANG MO KIO AVE 4     01 TO 03   
2  2017-01  ANG MO KIO    3 ROOM   602   ANG MO KIO AVE 5     01 TO 03   
3  2017-01  ANG MO KIO    3 ROOM   465  ANG MO KIO AVE 10     04 TO 06   
4  2017-01  ANG MO KIO    3 ROOM   601   ANG MO KIO AVE 5     01 TO 03   

   floor_area_sqm      flat_model  lease_commence_date     remaining_lease  \
0            44.0        Improved                 1979  61 years 04 months   
1            67.0  New Generation                 1978  60 years 07 months   
2            67.0  New Generation              

In [3]:
# Step 1: Convert the 'month' column in resale_df to datetime
print("\nConverting month to datetime format...")
resale_df['month'] = pd.to_datetime(resale_df['month'])


Converting month to datetime format...


In [4]:
# Step 2: Create a 'quarter' column in resale_df in format "YYYY-QX"
print("Creating quarter column from monthly data...")
resale_df['year'] = resale_df['month'].dt.year
resale_df['q'] = resale_df['month'].dt.quarter
resale_df['quarter'] = resale_df['year'].astype(str) + '-Q' + resale_df['q'].astype(str)
resale_df = resale_df.drop(columns=['year', 'q'])

Creating quarter column from monthly data...


In [5]:
# Step 3: Check the quarter format in both datasets
print("\nQuarter format in RPI dataset:")
print(rpi_df['quarter'].head())
print("\nQuarter format in Resale dataset:")
print(resale_df['quarter'].head())
print(f"Unique quarters in Resale dataset: {resale_df['quarter'].nunique()}")


Quarter format in RPI dataset:
0    1990-Q1
1    1990-Q2
2    1990-Q3
3    1990-Q4
4    1991-Q1
Name: quarter, dtype: object

Quarter format in Resale dataset:
0    2017-Q1
1    2017-Q1
2    2017-Q1
3    2017-Q1
4    2017-Q1
Name: quarter, dtype: object
Unique quarters in Resale dataset: 37


In [6]:
# Step 4: Merge the datasets
print("\nMerging datasets on 'quarter' column...")
merged_df = resale_df.merge(rpi_df, on='quarter', how='left')
merged_df = merged_df.rename(columns={'index': 'rpi'})

print(f"Merged dataset shape: {merged_df.shape}")


Merging datasets on 'quarter' column...
Merged dataset shape: (224541, 13)


In [ ]:
# Step 5: Adjust resale prices by RPI
# Set base period to 2017 Q1 
BASE_RPI = 133.9  # 2017 Q1 RPI value

# Create real prices column
merged_df['resale_price_real'] = merged_df['resale_price'] * (BASE_RPI / merged_df['rpi'])

# Round to 2 decimal places for cleaner data
merged_df['resale_price_real'] = merged_df['resale_price_real'].round(2)

# Remove rows without RPI data (can't calculate real prices for these)
print(f"\nRows before filtering: {len(merged_df):,}")
print(f"Rows with missing RPI: {merged_df['rpi'].isna().sum():,}")

merged_df = merged_df[merged_df['rpi'].notna()].copy()

print(f"Rows after filtering: {len(merged_df):,}")


Rows before filtering: 224,541
Rows with missing RPI: 2,464
Rows after filtering: 222,077


In [8]:
# Step 6: Check for missing RPI values (quarters not in RPI dataset)
missing_rpi = merged_df['rpi'].isna().sum()
print(f"\nNumber of records without RPI match: {missing_rpi}")
if missing_rpi > 0:
    print("Quarters without RPI data:")
    print(merged_df[merged_df['rpi'].isna()]['quarter'].unique())


Number of records without RPI match: 0


In [9]:
# Step 7: Display summary statistics
print("\n=== MERGE SUMMARY ===")
print(f"Total resale transactions: {len(merged_df):,}")
print(f"Transactions with RPI data: {len(merged_df[merged_df['rpi'].notna()]):,}")
print(f"Date range: {resale_df['month'].min()} to {resale_df['month'].max()}")
print(f"\nColumns in merged dataset:")
print(merged_df.columns.tolist())


=== MERGE SUMMARY ===
Total resale transactions: 222,077
Transactions with RPI data: 222,077
Date range: 2017-01-01 00:00:00 to 2026-02-01 00:00:00

Columns in merged dataset:
['month', 'town', 'flat_type', 'block', 'street_name', 'storey_range', 'floor_area_sqm', 'flat_model', 'lease_commence_date', 'remaining_lease', 'resale_price', 'quarter', 'rpi', 'resale_price_real']


In [10]:
# Step 8: Save the merged dataset
output_file = '../merged_data/merged_hdb_resale_with_rpi.csv'
print(f"\nSaving merged dataset to {output_file}...")
merged_df.to_csv(output_file, index=False)
print("Done!")


Saving merged dataset to ../merged_data/merged_hdb_resale_with_rpi.csv...
Done!


In [11]:
# Display first few rows of merged data
print("\nFirst 5 rows of merged dataset:")
print(merged_df.head())

# Display basic statistics
print("\nBasic statistics of resale prices and RPI:")
print(merged_df[['resale_price', 'rpi', 'resale_price_real']].describe())

# Optional: Create a summary by quarter
print("\n=== CREATING QUARTERLY SUMMARY ===")
quarterly_summary = merged_df.groupby('quarter').agg({
    'resale_price': ['count', 'mean', 'median', 'min', 'max'],
    'resale_price_real': ['mean', 'median'],  # Add real price statistics
    'rpi': 'first'  # RPI is the same for all transactions in a quarter
}).round(2)

quarterly_summary.columns = ['_'.join(col).strip() for col in quarterly_summary.columns.values]
quarterly_summary = quarterly_summary.reset_index()
quarterly_summary.columns = ['quarter', 'num_transactions', 'avg_price', 'median_price', 
                              'min_price', 'max_price', 'avg_real_price', 'median_real_price', 'rpi']

print("\nQuarterly summary (first 10 quarters):")
print(quarterly_summary.head(10))

# Save quarterly summary
summary_file = '../merged_data/quarterly_summary.csv'
quarterly_summary.to_csv(summary_file, index=False)
print(f"\nQuarterly summary saved to {summary_file}")


First 5 rows of merged dataset:
       month        town flat_type block        street_name storey_range  \
0 2017-01-01  ANG MO KIO    2 ROOM   406  ANG MO KIO AVE 10     10 TO 12   
1 2017-01-01  ANG MO KIO    3 ROOM   108   ANG MO KIO AVE 4     01 TO 03   
2 2017-01-01  ANG MO KIO    3 ROOM   602   ANG MO KIO AVE 5     01 TO 03   
3 2017-01-01  ANG MO KIO    3 ROOM   465  ANG MO KIO AVE 10     04 TO 06   
4 2017-01-01  ANG MO KIO    3 ROOM   601   ANG MO KIO AVE 5     01 TO 03   

   floor_area_sqm      flat_model  lease_commence_date     remaining_lease  \
0            44.0        Improved                 1979  61 years 04 months   
1            67.0  New Generation                 1978  60 years 07 months   
2            67.0  New Generation                 1980  62 years 05 months   
3            68.0  New Generation                 1980   62 years 01 month   
4            67.0  New Generation                 1980  62 years 05 months   

   resale_price  quarter    rpi  resale_p